# Lab 1: การตรวจสอบและลดอคติของแบบจำลอง (Fairness Audit & Mitigation)
**รายวิชา:** จริยธรรมในการใช้ AI และ Explainable AI
**เวลาโดยประมาณ:** 90–120 นาที | **Runtime:** CPU เพียงพอ

## วัตถุประสงค์
1. วัดความเป็นธรรมเชิงกลุ่ม (group fairness) ด้วย `MetricFrame` ของ Fairlearn
2. แสดงให้เห็นว่า *fairness through unawareness* (การตัดตัวแปรอ่อนไหวออก) ไม่เพียงพอ เนื่องจากมีตัวแปรแทน (proxy)
3. เปรียบเทียบวิธีลดอคติ 3 ระยะ: pre-processing (Reweighing), in-processing (Exponentiated Gradient), post-processing (Threshold Optimizer)
4. วิเคราะห์ trade-off ระหว่างความแม่นยำกับความเป็นธรรม และเขียนสรุปในรูปแบบ Model Card

**ชุดข้อมูล:** UCI Adult (Census Income) ผ่าน OpenML — ทำนายว่ารายได้ > 50K USD/ปี หรือไม่  
**ข้อควรระวังเชิงวิชาการ:** Adult เป็นข้อมูลปี 1994 และมีข้อจำกัดที่ถูกวิพากษ์ในวรรณกรรม จึงใช้เพื่อการสอนวิธีการเท่านั้น ไม่ใช่เพื่อสรุปข้อเท็จจริงทางสังคม


In [ ]:
!pip -q install fairlearn

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import cross_val_predict

from fairlearn.metrics import (MetricFrame, selection_rate, count,
                               true_positive_rate, false_positive_rate,
                               demographic_parity_difference, equalized_odds_difference)
from fairlearn.reductions import ExponentiatedGradient, DemographicParity, EqualizedOdds
from fairlearn.postprocessing import ThresholdOptimizer

RANDOM_STATE = 42
pd.set_option("display.float_format", "{:.4f}".format)

## ส่วนที่ 1 โหลดข้อมูลและสำรวจอัตราฐาน (base rate)
อัตราฐานที่ต่างกันระหว่างกลุ่ม คือเงื่อนไขสำคัญของทฤษฎีบทความเป็นไปไม่ได้ (Kleinberg et al., 2017; Chouldechova, 2017)

In [ ]:
adult = fetch_openml("adult", version=2, as_frame=True)
df = adult.frame.rename(columns={"class": "income"}).copy()
df["y"] = (df["income"].astype(str) == ">50K").astype(int)
print(df.shape)
df.head()

In [ ]:
print("อัตราฐาน P(Y=1) แยกตามเพศ")
display(df.groupby("sex", observed=True)["y"].agg(["mean", "count"]))
print("อัตราฐาน P(Y=1) แยกตามเชื้อชาติ")
display(df.groupby("race", observed=True)["y"].agg(["mean", "count"]))

## ส่วนที่ 2 แบ่งข้อมูล และสร้างแบบจำลองแบบ "ไม่รับรู้" ตัวแปรอ่อนไหว (unaware)
เราตัด `sex` และ `race` ออกจากตัวแปรอิสระ แต่เก็บ `sex` ไว้เป็น sensitive feature สำหรับการประเมิน

In [ ]:
X = df.drop(columns=["income", "y", "fnlwgt"])
y = df["y"]
A = df["sex"].astype(str)          # sensitive feature หลัก
A_race = df["race"].astype(str)    # สำหรับวิเคราะห์ intersectional

X_tr, X_te, y_tr, y_te, A_tr, A_te, R_tr, R_te = train_test_split(
    X, y, A, A_race, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

SENSITIVE_COLS = ["sex", "race"]
X_tr_u, X_te_u = X_tr.drop(columns=SENSITIVE_COLS), X_te.drop(columns=SENSITIVE_COLS)

def make_preprocessor(X):
    num = X.select_dtypes(include="number").columns.tolist()
    cat = [c for c in X.columns if c not in num]
    cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                         ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", StandardScaler(), num), ("cat", cat_pipe, cat)])

prep = make_preprocessor(X_tr_u)
Z_tr = prep.fit_transform(X_tr_u)
Z_te = prep.transform(X_te_u)
print("มิติหลัง one-hot:", Z_tr.shape)

In [ ]:
base = LogisticRegression(max_iter=3000)
base.fit(Z_tr, y_tr)
y_base = base.predict(Z_te)
print("Accuracy:", round(accuracy_score(y_te, y_base), 4),
      "| AUC:", round(roc_auc_score(y_te, base.predict_proba(Z_te)[:, 1]), 4))

## ส่วนที่ 3 วัดความเป็นธรรมด้วย MetricFrame
- **Demographic parity difference (DPD):** ผลต่างสูงสุดของ P(Ŷ=1 | A=a) ระหว่างกลุ่ม  
- **Equalized odds difference (EOD):** ค่าสูงสุดของผลต่าง TPR และ FPR ระหว่างกลุ่ม

In [ ]:
metrics = {"accuracy": accuracy_score, "selection_rate": selection_rate,
           "TPR": true_positive_rate, "FPR": false_positive_rate, "n": count}

mf = MetricFrame(metrics=metrics, y_true=y_te, y_pred=y_base, sensitive_features=A_te)
display(mf.by_group)
print("ผลต่างระหว่างกลุ่ม (max - min):")
display(mf.difference())
print("DPD =", round(demographic_parity_difference(y_te, y_base, sensitive_features=A_te), 4))
print("EOD =", round(equalized_odds_difference(y_te, y_base, sensitive_features=A_te), 4))

mf.by_group[["selection_rate", "TPR", "FPR"]].plot.bar(rot=0, figsize=(7, 4),
                                                        title="Baseline (unaware) by sex")
plt.tight_layout(); plt.show()

### 3.1 Intersectional analysis
การวิเคราะห์เพศเพียงมิติเดียวอาจซ่อนความเหลื่อมล้ำของกลุ่มย่อย (Buolamwini & Gebru, 2018) ให้สังเกตขนาดตัวอย่าง `n` ของแต่ละกลุ่มก่อนตีความ

In [ ]:
mf_int = MetricFrame(metrics=metrics, y_true=y_te, y_pred=y_base,
                     sensitive_features=pd.DataFrame({"sex": A_te, "race": R_te}))
display(mf_int.by_group.sort_values("selection_rate"))

## ส่วนที่ 4 ทำไม unawareness จึงไม่พอ: ทดสอบตัวแปรแทน (proxy test)
ถ้าตัวแปรที่เหลือทำนาย `sex` ได้ดี (AUC สูงกว่า 0.5 มาก) แสดงว่าข้อมูลเพศยังแฝงอยู่ในตัวแปรอื่น เช่น `relationship`, `marital-status`, `occupation`

In [ ]:
proxy_clf = HistGradientBoostingClassifier(random_state=RANDOM_STATE)
a_bin = (A_tr == "Male").astype(int)
p_hat = cross_val_predict(proxy_clf, Z_tr, a_bin, cv=5, method="predict_proba")[:, 1]
print("AUC ในการทำนายเพศจากตัวแปรที่เหลือ:", round(roc_auc_score(a_bin, p_hat), 4))

## ส่วนที่ 5 การลดอคติ 3 ระยะ
### 5.1 Pre-processing: Reweighing (Kamiran & Calders, 2012)
กำหนดน้ำหนัก $w(a,y) = \dfrac{P(A=a)\,P(Y=y)}{P(A=a, Y=y)}$ เพื่อให้ A และ Y เป็นอิสระต่อกันในข้อมูลที่ถ่วงน้ำหนักแล้ว

In [ ]:
def reweighing_weights(a, yv):
    d = pd.DataFrame({"a": np.asarray(a), "y": np.asarray(yv)})
    p_a = d["a"].value_counts(normalize=True)
    p_y = d["y"].value_counts(normalize=True)
    p_ay = d.value_counts(normalize=True)
    return d.apply(lambda r: p_a[r.a] * p_y[r.y] / p_ay[(r.a, r.y)], axis=1).to_numpy()

w_tr = reweighing_weights(A_tr, y_tr)
rw = LogisticRegression(max_iter=3000).fit(Z_tr, y_tr, sample_weight=w_tr)
y_rw = rw.predict(Z_te)

### 5.2 In-processing: Exponentiated Gradient (Agarwal et al., 2018)
แก้ปัญหา $\min_h \text{err}(h)$ ภายใต้เงื่อนไข $|\text{DP gap}| \le \varepsilon$ ผ่านเกมลากรานจ์ ผลลัพธ์เป็น randomized classifier  
*(ใช้เวลาประมาณ 1–3 นาทีบน CPU)*

In [ ]:
eg = ExponentiatedGradient(LogisticRegression(max_iter=3000),
                           constraints=DemographicParity(), eps=0.01)
eg.fit(Z_tr, y_tr, sensitive_features=A_tr)
y_eg = eg.predict(Z_te, random_state=RANDOM_STATE)

### 5.3 Post-processing: Threshold Optimizer (Hardt et al., 2016)
เลือกเกณฑ์ตัดสินแยกตามกลุ่มเพื่อให้เป็นไปตาม equalized odds โดยไม่ฝึกแบบจำลองใหม่  
**ประเด็นอภิปราย:** วิธีนี้ต้องใช้ A ขณะทำนาย ซึ่งอาจขัดกับกฎหมายในบางบริบท

In [ ]:
to = ThresholdOptimizer(estimator=base, constraints="equalized_odds",
                        objective="balanced_accuracy_score", prefit=True,
                        predict_method="predict_proba")
to.fit(Z_tr, y_tr, sensitive_features=A_tr)
y_to = to.predict(Z_te, sensitive_features=A_te, random_state=RANDOM_STATE)

## ส่วนที่ 6 เปรียบเทียบผลลัพธ์

In [ ]:
def evaluate(name, yp):
    return {"method": name,
            "accuracy": accuracy_score(y_te, yp),
            "balanced_acc": balanced_accuracy_score(y_te, yp),
            "DPD": demographic_parity_difference(y_te, yp, sensitive_features=A_te),
            "EOD": equalized_odds_difference(y_te, yp, sensitive_features=A_te)}

results = pd.DataFrame([evaluate("Baseline (unaware)", y_base),
                        evaluate("Reweighing (pre)", y_rw),
                        evaluate("ExpGrad-DP (in)", y_eg),
                        evaluate("ThresholdOpt-EO (post)", y_to)]).set_index("method")
display(results)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for m, r in results.iterrows():
    ax[0].scatter(r["DPD"], r["accuracy"], s=60); ax[0].annotate(m, (r["DPD"], r["accuracy"]), fontsize=8)
    ax[1].scatter(r["EOD"], r["accuracy"], s=60); ax[1].annotate(m, (r["EOD"], r["accuracy"]), fontsize=8)
ax[0].set_xlabel("DPD (lower is better)"); ax[1].set_xlabel("EOD (lower is better)")
for a in ax: a.set_ylabel("Accuracy"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## ส่วนที่ 7 (ขั้นสูง) Pareto frontier ของ accuracy–fairness
ปรับค่า `eps` ของ Exponentiated Gradient เพื่อดูเส้นขอบ trade-off *(อาจใช้เวลาหลายนาที ลดจำนวนค่า eps ได้)*

In [ ]:
frontier = []
for eps in [0.005, 0.02, 0.05, 0.1]:
    m = ExponentiatedGradient(LogisticRegression(max_iter=3000),
                              constraints=DemographicParity(), eps=eps)
    m.fit(Z_tr, y_tr, sensitive_features=A_tr)
    r = evaluate(f"eps={eps}", m.predict(Z_te, random_state=RANDOM_STATE)); frontier.append(r)
frontier = pd.DataFrame(frontier)
display(frontier)
plt.plot(frontier["DPD"], frontier["accuracy"], "o-")
for _, r in frontier.iterrows(): plt.annotate(r["method"], (r["DPD"], r["accuracy"]), fontsize=8)
plt.xlabel("DPD"); plt.ylabel("Accuracy"); plt.title("Accuracy–fairness frontier"); plt.grid(alpha=.3); plt.show()

## ส่วนที่ 8 สรุปแบบ Model Card (Mitchell et al., 2019)
เติมข้อความในเซลล์ถัดไปให้ครบ แล้วแนบในรายงาน

In [ ]:
best = results.sort_values("DPD").index[0]
card = f'''
### Model Card (ร่าง)
- **Intended use:** [ระบุวัตถุประสงค์และผู้ใช้]
- **Out-of-scope use:** [ระบุการใช้งานที่ไม่ควรทำ]
- **Training data:** UCI Adult (OpenML v2), n_train={len(y_tr)}; ข้อจำกัด: [ระบุ]
- **Sensitive attributes evaluated:** sex (หลัก), sex x race (intersectional)
- **Fairness metrics (test):** ดูตารางผลด้านบน; วิธีที่ DPD ต่ำสุด = {best}
- **Trade-offs:** [อภิปรายการเสีย accuracy เทียบกับการลด DPD/EOD]
- **Ethical considerations:** [proxy, การใช้ A ขณะทำนาย, บริบททางกฎหมาย]
'''
print(card)

## แบบฝึกหัดสำหรับรายงาน
1. เปลี่ยน constraint ของ Exponentiated Gradient เป็น `EqualizedOdds()` แล้วเปรียบเทียบ DPD/EOD กับกรณี `DemographicParity()` อธิบายผลด้วยทฤษฎีบทความเป็นไปไม่ได้
2. ประเมินผลด้วย bootstrap (≥ 200 รอบ) เพื่อรายงานช่วงความเชื่อมั่น 95% ของ DPD และ EOD
3. สำหรับบริบท "สินเชื่อ" และ "การคัดกรองโรค" ท่านจะเลือก fairness metric ใด เพราะเหตุใด (อ้างอิงอย่างน้อย 2 แหล่ง)
4. เสนอสมมติฐานการวิจัย 1 ข้อเกี่ยวกับ proxy discrimination และออกแบบการทดลองเพื่อทดสอบ
